# Harmonic downgrading: real d10 dust template

Comparison of five downgrade methods on the PySM d10 GNILC dust intensity
template (353 GHz, NSIDE 2048 → 128).  Unlike the synthetic benchmarks,
there is no independent ground truth — diagnostics are relative.

## Method overview

The harmonic downgrade follows **Planck 2015 XVI Eq. 1**:

$$
a^{\text{out}}_{\ell m}
= \frac{p^{\text{out}}_\ell}{p^{\text{in}}_\ell}
  \;\frac{b^{\text{out}}_\ell}{b^{\text{in}}_\ell}
  \; a^{\text{in}}_{\ell m}
$$

| # | Method | Description |
|---|--------|-------------|
| 1 | **ud_grade** | Pixel averaging (baseline) |
| 2 | **Harmonic clip** | SHT truncation only |
| 3 | **Harmonic + pixwin** | + pixel window correction |
| 4 | **Harmonic + pixwin + beam** | Full Eq. 1 (recommended) |
| 5 | **Smoothing + ud_grade** | Traditional approach |

In [ ]:
%matplotlib inline
import gc
from pathlib import Path
import urllib.request

import numpy as np
import healpy as hp
import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import skew, kurtosis

matplotlib.rcParams.update({'font.size': 11, 'axes.grid': True, 'grid.alpha': 0.3})

COLORS = {
    'ud_grade':   '#d95f02',
    'clip':       '#1b9e77',
    'pixwin':     '#33a02c',
    'full':       '#1f78b4',
    'smooth_ud':  '#7570b3',
}
METHOD_LABELS = {
    'ud_grade':   'ud_grade',
    'clip':       'harmonic (clip)',
    'pixwin':     'harmonic + pixwin',
    'full':       'harmonic + pixwin + beam',
    'smooth_ud':  'smoothing + ud_grade',
}
METHOD_ORDER = ['ud_grade', 'clip', 'pixwin', 'full', 'smooth_ud']
LINESTYLES = {
    'ud_grade':   '-',
    'clip':       '--',
    'pixwin':     '-.',
    'full':       '-',
    'smooth_ud':  ':',
}

## Load d10 template and downgrade

The d10 GNILC dust intensity template is read at its native NSIDE = 2048.
All downgrade methods and pre-computations that need the full-resolution
map are performed in this cell, after which the large array is freed.

In [ ]:
base_url = 'https://portal.nersc.gov/project/cmb/pysm-data'
fname = 'gnilc_dust_template_nside2048_2023.02.10.fits'
local_dir = Path('/tmp/pysm_data')
local_dir.mkdir(parents=True, exist_ok=True)
local_path = local_dir / fname

if not local_path.exists():
    print(f'Downloading {fname}...')
    urllib.request.urlretrieve(f'{base_url}/dust_gnilc/{fname}', local_path)
    print('Done.')
else:
    print(f'Using cached {local_path}')

m_d10 = hp.read_map(local_path, field=0, dtype=np.float64)
print(f"d10: NSIDE={hp.get_nside(m_d10)}, npix={len(m_d10)}, "
      f"range=[{np.nanmin(m_d10):.2e}, {np.nanmax(m_d10):.2e}]")

In [ ]:
nside_in = hp.get_nside(m_d10)
nside_out = 128
lmax_out = 3 * nside_out - 1  # 383

# Pixel window functions
pw_in = hp.pixwin(nside_in, lmax=lmax_out)
pw_out = hp.pixwin(nside_out, lmax=lmax_out)

# Pre-compute spectrum of original at output lmax
cl_orig = hp.anafast(m_d10, lmax=lmax_out)

# Pre-compute pixel statistics of the full-resolution map
finite = m_d10[np.isfinite(m_d10)]
orig_stats = {
    'mean': float(np.mean(finite)),
    'std': float(np.std(finite)),
    'skew': float(skew(finite)),
    'kurtosis': float(kurtosis(finite)),
}
del finite

# Output beam FWHM (auto-computed by harmonic_ud_grade)
fwhm_auto = 3 * hp.nside2resol(nside_out)

# ---------- Downgrade methods ----------
maps = {}

# 1. ud_grade (pixel averaging)
maps['ud_grade'] = hp.ud_grade(m_d10, nside_out)

# 2. Harmonic clip (SHT truncation only)
maps['clip'] = hp.harmonic_ud_grade(
    m_d10, nside_out, pixwin=False, fwhm_out=0,
)

# 3. Harmonic + pixwin (pixel window correction, no beam)
maps['pixwin'] = hp.harmonic_ud_grade(
    m_d10, nside_out, pixwin=True, fwhm_out=0,
)

# 4. Harmonic + pixwin + beam (full Eq. 1)
maps['full'] = hp.harmonic_ud_grade(m_d10, nside_out)

# 5. Smoothing + ud_grade (traditional)
m_smooth = hp.smoothing(m_d10, fwhm=fwhm_auto, lmax=lmax_out)
maps['smooth_ud'] = hp.ud_grade(m_smooth, nside_out)
del m_smooth

# ---------- Free memory ----------
del m_d10
gc.collect()

print(f'NSIDE_out = {nside_out}, lmax_out = {lmax_out}')
print(f'Output beam FWHM = {np.degrees(fwhm_auto):.2f} deg '
      f'({np.degrees(fwhm_auto) * 60:.1f} arcmin)')
print('Memory freed: full-resolution map deleted')
for key in METHOD_ORDER:
    print(f'  {METHOD_LABELS[key]}: shape={maps[key].shape}, '
          f'range=[{np.nanmin(maps[key]):.2e}, {np.nanmax(maps[key]):.2e}]')

## Mollweide maps

In [ ]:
all_vals = np.concatenate([maps[k] for k in METHOD_ORDER])
vmin, vmax = np.nanpercentile(all_vals, [1, 99])
del all_vals

span = vmax - vmin
tick = 10 ** np.floor(np.log10(span / 5))
vmin = float(np.floor(vmin / tick) * tick)
vmax = float(np.ceil(vmax / tick) * tick)
print(f'Color range: [{vmin:.1f}, {vmax:.1f}]')

fig = plt.figure(figsize=(28, 5))
for i, key in enumerate(METHOD_ORDER):
    hp.projview(
        maps[key],
        fig=fig,
        sub=(1, 5, i + 1),
        title=METHOD_LABELS[key],
        min=vmin,
        max=vmax,
        cmap='viridis',
        cb_orientation='horizontal',
        unit=r'$\mu K_{RJ}$',
    )
fig.suptitle(f'd10 dust 353 GHz — downgraded to NSIDE {nside_out}', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Residual maps (method − full Eq. 1)

Since there is no independent reference at NSIDE 128, residuals are
computed relative to the **full Eq. 1** result (pixwin + beam).
These residuals show what changes when parts of the correction are omitted.

In [ ]:
residual_keys = ['ud_grade', 'clip', 'pixwin', 'smooth_ud']
residuals = {key: maps[key] - maps['full'] for key in residual_keys}

residual_titles = {
    'ud_grade':  'ud_grade − full',
    'clip':      'clip − full',
    'pixwin':    'pixwin − full',
    'smooth_ud': 'smooth+ud − full',
}

fig = plt.figure(figsize=(24, 5))
for i, key in enumerate(residual_keys):
    vlim = float(np.nanpercentile(np.abs(residuals[key]), 99))
    hp.projview(
        residuals[key],
        fig=fig,
        sub=(1, 4, i + 1),
        title=residual_titles[key],
        min=-vlim,
        max=vlim,
        cmap='RdBu_r',
        cb_orientation='horizontal',
        unit=r'$\mu K_{RJ}$',
    )
fig.suptitle('Residuals relative to full Eq. 1', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Gnomonic zoom

Close-ups at the **north Galactic pole** (diffuse, faint emission) and
the **Galactic centre** (bright, structured emission).
At NSIDE 128 the pixel size is ≈ 27′.

In [ ]:
locations = [
    ('North Galactic Pole (b=90\u00b0)', (0, 90)),
    ('Galactic Centre (l=0\u00b0, b=0\u00b0)', (0, 0)),
]

for loc_name, rot in locations:
    # Compute shared color range from pixels near the target
    vec = hp.ang2vec(rot[0], rot[1], lonlat=True)
    ipix = hp.query_disc(nside_out, vec, np.radians(5))
    vals = np.concatenate([maps[k][ipix] for k in METHOD_ORDER])
    loc_vmin = float(np.nanpercentile(vals, 1))
    loc_vmax = float(np.nanpercentile(vals, 99))
    del vals

    fig = plt.figure(figsize=(26, 5))
    fig.suptitle(loc_name, fontsize=14)
    for i, key in enumerate(METHOD_ORDER):
        hp.gnomview(
            maps[key],
            fig=fig,
            sub=(1, 5, i + 1),
            rot=rot,
            reso=2,
            xsize=200,
            min=loc_vmin,
            max=loc_vmax,
            cmap='viridis',
            title=METHOD_LABELS[key],
            notext=True,
        )
    plt.tight_layout()
    plt.show()

## Power spectra (pixel-window deconvolved)

The black line is the spectrum of the original NSIDE 2048 map computed up
to $\ell_{\max}^{\mathrm{out}} = 383$.  All spectra are divided by
$p_\ell^2(N_{\mathrm{side,out}})$ to show the underlying spectral shape.

**Why does ud_grade appear elevated at high $\ell$?**
Unlike the synthetic power-law benchmark (where the input spectrum falls
steeply), real dust emission has substantial power at $\ell > \ell_{\max}^{\mathrm{out}}$.
`ud_grade` performs pixel averaging without any bandlimit — this **aliases**
high-$\ell$ power into the output band, boosting the measured spectrum.
In the power-law notebook the input spectrum already falls so steeply that
the aliased contribution is negligible; here the flatter dust spectrum
makes the effect dramatic.

In [ ]:
ell = np.arange(len(cl_orig))

cl = {}
for key in METHOD_ORDER:
    cl[key] = hp.anafast(maps[key], lmax=lmax_out)

fig, ax = plt.subplots(figsize=(10, 6))

ax.loglog(ell[1:], (cl_orig / pw_in**2)[1:], 'k-', linewidth=2,
          label=rf'original (NSIDE {nside_in})', zorder=10)

for key in METHOD_ORDER:
    ax.loglog(ell[1:], (cl[key] / pw_out**2)[1:],
              color=COLORS[key], ls=LINESTYLES[key],
              label=METHOD_LABELS[key], linewidth=1.5)

ax.set_xlabel(r'$\ell$')
ax.set_ylabel(r'$C_\ell \;/\; p_\ell^2$')
ax.set_title('Power spectra (pixel-window deconvolved)')
ax.legend(fontsize=9)
ax.set_xlim(1, lmax_out)
plt.tight_layout()
plt.show()

## Fractional spectral difference vs original

$(C_\ell^{\mathrm{method}} - C_\ell^{\mathrm{orig}}) / C_\ell^{\mathrm{orig}}$,
where both are pixel-window-deconvolved.  The grey dotted curve shows
$p_\ell^2 - 1$.

In [ ]:
cl_ref = cl_orig / pw_in**2

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

panels = [
    (2, lmax_out,
     rf'Full range ($\ell$ = 2\u2013{lmax_out})'),
    (lmax_out // 2, lmax_out,
     rf'High $\ell$ ($\ell$ = {lmax_out // 2}\u2013{lmax_out})'),
]

for ax, (ell_min, ell_max, title) in zip(axes, panels):
    mask = (ell >= ell_min) & (ell <= ell_max) & (cl_ref > 0)

    ax.plot(ell[mask], (pw_out**2 - 1)[mask], 'grey', linestyle=':',
            linewidth=1.5, label=r'$p_\ell^2 - 1$', zorder=0)

    for key in METHOD_ORDER:
        cl_deconv = cl[key] / pw_out**2
        frac = (cl_deconv - cl_ref) / cl_ref
        ax.plot(ell[mask], frac[mask], color=COLORS[key],
                ls=LINESTYLES[key],
                label=METHOD_LABELS[key], linewidth=1.5)

    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_xlabel(r'$\ell$')
    ax.set_ylabel(
        r'$(C_\ell^{\mathrm{method}} - C_\ell^{\mathrm{orig}})'
        r' / C_\ell^{\mathrm{orig}}$'
    )
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.suptitle('Fractional spectral difference vs original', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Pixel-space statistics

One-point statistics of each downgraded map compared to the original
NSIDE 2048 map.  Differences in mean, std, skewness, and kurtosis reveal
how each method affects the pixel value distribution.

In [ ]:
method_stats = {}
for key in METHOD_ORDER:
    vals = maps[key][np.isfinite(maps[key])]
    method_stats[key] = {
        'mean': float(np.mean(vals)),
        'std': float(np.std(vals)),
        'skew': float(skew(vals)),
        'kurtosis': float(kurtosis(vals)),
    }

print(f"{'Method':<30} {'Mean':>12} {'Std':>12} {'Skew':>10} {'Kurtosis':>10}")
print('-' * 74)
print(f"{'Original (NSIDE ' + str(nside_in) + ')':<30} {orig_stats['mean']:>12.4f} "
      f"{orig_stats['std']:>12.4f} {orig_stats['skew']:>10.4f} "
      f"{orig_stats['kurtosis']:>10.4f}")
for key in METHOD_ORDER:
    s = method_stats[key]
    print(f"{METHOD_LABELS[key]:<30} {s['mean']:>12.4f} {s['std']:>12.4f} "
          f"{s['skew']:>10.4f} {s['kurtosis']:>10.4f}")

stat_defs = [
    (r'Mean shift / $\sigma_{\mathrm{orig}}$',
     lambda s: (s['mean'] - orig_stats['mean']) / orig_stats['std']),
    (r'$(\sigma - \sigma_{\mathrm{orig}}) / \sigma_{\mathrm{orig}}$',
     lambda s: (s['std'] - orig_stats['std']) / orig_stats['std']),
    ('Skewness difference',
     lambda s: s['skew'] - orig_stats['skew']),
    ('Kurtosis difference',
     lambda s: s['kurtosis'] - orig_stats['kurtosis']),
]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (ylabel, func) in zip(axes, stat_defs):
    values = [func(method_stats[k]) for k in METHOD_ORDER]
    ax.bar(
        range(len(METHOD_ORDER)), values,
        color=[COLORS[k] for k in METHOD_ORDER],
    )
    ax.set_ylabel(ylabel)
    ax.set_xticks(range(len(METHOD_ORDER)))
    ax.set_xticklabels(
        [METHOD_LABELS[k] for k in METHOD_ORDER],
        rotation=45, ha='right', fontsize=8,
    )
    ax.axhline(0, color='black', linewidth=0.5)

fig.suptitle('Pixel statistics: deviations from original', fontsize=14)
plt.tight_layout()
plt.show()

## Takeaways

| Observation | Implication |
|---|---|
| **ud_grade** preserves one-point statistics well but introduces spectral aliasing | Acceptable for pixel-space analyses, not for spectral work |
| **Clip-only** removes aliasing but has pixwin bias at high ℓ | Insufficient for precision spectral analyses |
| **Pixwin correction** fixes the spectral bias | Important for any spectral analysis |
| **Full Eq. 1** (pixwin + beam) gives the cleanest spectra | Recommended default for most applications |
| **Smoothing + ud_grade** behaves similarly to full Eq. 1 | A simpler alternative, but less principled |
| The beam suppresses power at high ℓ, reducing std and increasing smoothness | Expected and desirable for map visualisation |